# Segmentación de imagen con YOLOv8

Este notebook:
1. Instala/carga `ultralytics` (YOLOv8).
2. El usuario carga una imagen RGB.
3. Corre el modelo de **segmentación** de YOLOv8 (`yolov8n-seg.pt`) sobre la imagen.
4. Muestra y guarda la imagen segmentada (con máscaras, cajas y etiquetas de cada elemento detectado).


## 1. Instalar e importar librerías

Si `ultralytics` no está instalado en el entorno, la siguiente celda lo instala
(solo es necesario ejecutarla una vez).

In [ ]:
import sys
!{sys.executable} -m pip install -q ultralytics


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO


## 2. Cargar el modelo YOLOv8 de segmentación

Modelos disponibles (de menor a mayor tamaño/precisión):
`yolov8n-seg.pt`, `yolov8s-seg.pt`, `yolov8m-seg.pt`, `yolov8l-seg.pt`, `yolov8x-seg.pt`.

La primera vez que se ejecuta, el modelo se descarga automáticamente.

In [ ]:
MODELO = "yolov8n-seg.pt"  # <-- cambia el tamaño del modelo si lo necesitas

modelo = YOLO(MODELO)


## 3. Cargar la imagen del usuario

Modifica `RUTA_IMAGEN` con la ruta de tu imagen (jpg, png, etc.).

In [ ]:
RUTA_IMAGEN = "imagen.jpg"  # <-- cambia esta ruta por la de tu imagen

def cargar_imagen_rgb(ruta):
    """Carga una imagen desde disco y la devuelve como arreglo numpy en formato RGB (H, W, 3)."""
    img = Image.open(ruta).convert("RGB")
    return np.array(img)

imagen_original = cargar_imagen_rgb(RUTA_IMAGEN)
print(f"Dimensiones de la imagen: {imagen_original.shape}")


## 4. Ejecutar la segmentación

`modelo.predict` corre la inferencia y devuelve, entre otras cosas, las máscaras de segmentación
de cada objeto detectado, sus cajas delimitadoras, clases y niveles de confianza.

In [ ]:
CONFIANZA_MINIMA = 0.25  # umbral de confianza para mostrar una detección

resultados = modelo.predict(source=imagen_original, conf=CONFIANZA_MINIMA, verbose=False)
resultado = resultados[0]

if resultado.masks is None:
    print("No se detectó ningún elemento en la imagen.")
else:
    n_objetos = len(resultado.masks)
    print(f"Elementos segmentados: {n_objetos}")
    for i in range(n_objetos):
        clase_id = int(resultado.boxes.cls[i])
        nombre_clase = modelo.names[clase_id]
        confianza = float(resultado.boxes.conf[i])
        print(f"  {i+1}. {nombre_clase} (confianza: {confianza:.2f})")


## 5. Obtener la imagen segmentada

`resultado.plot()` dibuja las máscaras, cajas y etiquetas sobre la imagen y la devuelve
como un arreglo numpy en formato BGR (convención de OpenCV), por lo que se convierte a RGB.

In [ ]:
imagen_segmentada_bgr = resultado.plot()          # imagen con máscaras/cajas dibujadas (BGR)
imagen_segmentada = imagen_segmentada_bgr[:, :, ::-1]  # convertir BGR -> RGB


## 6. Subplot: imagen original vs. imagen segmentada

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 7))

ejes[0].imshow(imagen_original)
ejes[0].set_title("Imagen original")
ejes[0].axis("off")

ejes[1].imshow(imagen_segmentada)
ejes[1].set_title("Imagen segmentada (YOLOv8)")
ejes[1].axis("off")

plt.tight_layout()
plt.show()


## 7. (Opcional) Guardar la imagen segmentada en disco

In [ ]:
RUTA_SALIDA = "imagen_segmentada.png"

Image.fromarray(imagen_segmentada).save(RUTA_SALIDA)
print(f"Imagen segmentada guardada en: {RUTA_SALIDA}")
